# Spark-X2.6-4B — on-device agent on a free T4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/notebooks/blob/main/colab/Spark-X2.6-4B-on-device-agent.ipynb)

[`XHToken/Spark-X2.6-4B`](https://huggingface.co/XHToken/Spark-X2.6-4B) is SparkLLM's 4.11B on-device agent model (Apache 2.0): same `spark2_5` architecture as Spark-X2.5-4B, with a post-train pass aimed at coding, tool use, and short reasoning. The chat template takes `enable_thinking` plus `reasoning_effort` (`low` / `high`) and a `tools` list in OpenAI function form. The Hub checkpoint is about 8.2 GB in float16.

**Licence:** Apache 2.0.

**Runtime:** *Runtime > Change runtime type > T4 GPU*. The card pins `transformers==4.57.1` and loads with `trust_remote_code=True` (`modeling_spark.py`). `torch_dtype="auto"` is bfloat16; a T4 has no native bfloat16, so this notebook loads **float16** first. Watch for overflow on that path — if load or generate blows up, the load cell falls back to **4-bit** bitsandbytes. The shipped modeling code uses eager attention only, so FlashAttention 2 is not on this path. The card's `max_new_tokens=131072` does not fit a free T4; every demo stays in the low hundreds.

**Links:** [model card](https://huggingface.co/XHToken/Spark-X2.6-4B) · [GGUF](https://huggingface.co/XHToken/Spark-X2.6-4B-GGUF) (`Q4_K_M` ~3.54 GB)



In [ ]:
%pip install -q "transformers==4.57.1" "accelerate>=1.1.0" "bitsandbytes>=0.45.0"

# The model card requires exactly transformers==4.57.1 and trust_remote_code.
# Colab ships a newer transformers; this pin downgrades it. accelerate is for
# device_map="auto". bitsandbytes is the 4-bit fallback if float16 overflows
# on the T4. Pick Runtime > Change runtime type > T4 GPU before continuing.



> **Restart the runtime now:** *Runtime > Restart session*, then continue from the next cell.
> Colab has already imported its preinstalled transformers. The pin to 4.57.1 is invisible until the process restarts.



In [ ]:
import torch
import transformers
from packaging.version import Version

assert Version(transformers.__version__) == Version("4.57.1"), (
    f"transformers {transformers.__version__} is loaded; Spark-X2.6-4B's card pins 4.57.1. "
    "Run the install cell, then Runtime > Restart session."
)

if not torch.cuda.is_available():
    raise SystemExit("Select Runtime > Change runtime type > T4 GPU, then rerun from the top.")

# modeling_spark.py implements eager attention only. Keep SDPA on the
# mem-efficient and math kernels in case a later generate path touches them.
torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_math_sdp(True)

major, minor = torch.cuda.get_device_capability()
DTYPE = torch.bfloat16 if major >= 8 else torch.float16
print("transformers", transformers.__version__)
print("torch", torch.__version__)
print("GPU:", torch.cuda.get_device_name(), f"sm_{major}{minor}", "| dtype:", DTYPE)
print(f"VRAM total: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")



## 1. Load Spark-X2.6-4B

`AutoModelForCausalLM` and `AutoTokenizer`, as on the [model card](https://huggingface.co/XHToken/Spark-X2.6-4B#transformers). `trust_remote_code=True` is required: the checkpoint is `Spark2_5ForCausalLM`, not a stock transformers class. The first run downloads about 8.2 GB.

The card's generate snippet uses temperature 1.0, top_p 0.95, top_k 0. The saved `generation_config.json` stores `top_k: -1`, which transformers rejects, so the helper below always passes `top_k=0`.



In [ ]:
import gc
import re

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "XHToken/Spark-X2.6-4B"


def vram(label="allocated"):
    print(f"VRAM {label}: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")


def load_spark(four_bit=False):
    kwargs = {"trust_remote_code": True, "device_map": "auto"}
    if four_bit:
        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=DTYPE,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
        )
    else:
        kwargs["torch_dtype"] = DTYPE
    loaded = AutoModelForCausalLM.from_pretrained(MODEL_ID, **kwargs).eval()
    loaded.generation_config.top_k = 0
    return loaded


def reload_four_bit(reason):
    global model, LOAD_MODE
    print(reason)
    del model
    gc.collect()
    torch.cuda.empty_cache()
    model = load_spark(four_bit=True)
    LOAD_MODE = "4-bit"
    print("loaded 4-bit")
    vram()


tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
LOAD_MODE = "float16" if DTYPE == torch.float16 else "bfloat16"
try:
    model = load_spark(four_bit=False)
    print(f"loaded {LOAD_MODE}")
except Exception as exc:
    model = None
    reload_four_bit(f"{LOAD_MODE} load failed, retrying 4-bit: {exc}")

print("parameters:", f"{sum(p.numel() for p in model.parameters()) / 1e9:.2f}B")
print("load mode:", LOAD_MODE, "| dtype:", getattr(model, "dtype", None))
vram()


def split_think(text):
    if "</think>" not in text:
        return "", text.strip()
    think, answer = text.split("</think>", 1)
    return think.replace("<think>", "").strip(), answer.strip()


def generate_ids(inputs, max_new_tokens, temperature):
    with torch.inference_mode():
        return model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=0.95,
            top_k=0,
        )


def chat(messages, *, tools=None, enable_thinking=False, reasoning_effort="low", max_new_tokens=256, temperature=1.0):
    template = {
        "tokenize": True,
        "add_generation_prompt": True,
        "return_dict": True,
        "return_tensors": "pt",
        "enable_thinking": enable_thinking,
    }
    if enable_thinking:
        template["reasoning_effort"] = reasoning_effort
    if tools is not None:
        template["tools"] = tools
    inputs = tokenizer.apply_chat_template(messages, **template)
    device = next(model.parameters()).device
    inputs = {key: value.to(device) if hasattr(value, "to") else value for key, value in inputs.items()}
    try:
        generated = generate_ids(inputs, max_new_tokens, temperature)
    except RuntimeError as exc:
        overflow = any(token in str(exc).lower() for token in ("overflow", "nan", "inf"))
        if LOAD_MODE == "4-bit" or not overflow:
            raise
        reload_four_bit(f"float16 generate overflowed, reloading 4-bit: {exc}")
        device = next(model.parameters()).device
        inputs = {key: value.to(device) if hasattr(value, "to") else value for key, value in inputs.items()}
        generated = generate_ids(inputs, max_new_tokens, temperature)
    text = tokenizer.decode(generated[0, inputs["input_ids"].shape[-1] :], skip_special_tokens=True)
    vram()
    return text



## 2. Think-then-answer chat

`enable_thinking=True` opens a `<think>` channel. `reasoning_effort="low"` is enough for a short arithmetic check and keeps the trace well below the card's 131072-token cap. The printed block splits the think channel from the answer.



In [ ]:
think_messages = [{
    "role": "user",
    "content": (
        "A free T4 has 16 GB of VRAM. Spark-X2.6-4B has 4.11B parameters. "
        "About how many gigabytes is that in float16, and does it fit this GPU? "
        "Think briefly, then answer in two sentences."
    ),
}]
think_text = chat(think_messages, enable_thinking=True, reasoning_effort="low", max_new_tokens=320)
think, answer = split_think(think_text)
print("----- think -----")
print(think or "(no think channel)")
print("----- answer -----")
print(answer or think_text)



## 3. Coding task

Ask for one small parser, then run the generated function in a later cell against a fixed test file. The generate cell only writes source; it does not execute it.



In [ ]:
CODE_PROMPT = '''Write one Python function named parse_kv and nothing else.

Signature:
    def parse_kv(text: str) -> dict[str, str]:

Rules:
- each non-empty line is key=value
- skip blank lines and lines whose first non-space character is #
- strip whitespace around the key and the value
- if a key appears twice, the last assignment wins
- do not import anything; do not add a main block

Put the full function in one python code fence.'''

code_messages = [{"role": "user", "content": CODE_PROMPT}]
code_text = chat(code_messages, enable_thinking=False, max_new_tokens=400, temperature=0.4)
_, code_body = split_think(code_text)
fences = re.findall(r"```(?:python)?\n(.*?)```", code_body, flags=re.DOTALL)
SOLUTION_SRC = (fences[-1] if fences else code_body).strip() + "\n"
print(SOLUTION_SRC)



The next cell writes that source to a temp file, rejects a short list of dangerous imports, and runs `python -I` against four assertions. Isolated mode (`-I`) ignores `PYTHONPATH` and user site-packages.



In [ ]:
import ast
import subprocess
import tempfile
from pathlib import Path

TESTS = '''
from solution import parse_kv

assert parse_kv("host=localhost\nport=8080") == {"host": "localhost", "port": "8080"}
assert parse_kv("# comment\nname=a\nname=b\n") == {"name": "b"}
assert parse_kv("  token = abc  \n\n") == {"token": "abc"}
assert parse_kv("") == {}
print("all tests passed")
'''

BANNED = {"os", "subprocess", "socket", "pathlib", "shutil", "sys", "ctypes", "importlib", "requests", "urllib", "http", "pickle"}


def check_source(source):
    tree = ast.parse(source)
    for node in ast.walk(tree):
        if isinstance(node, ast.Import):
            for alias in node.names:
                root = alias.name.split(".")[0]
                if root in BANNED:
                    raise SystemExit(f"blocked import: {alias.name}")
        elif isinstance(node, ast.ImportFrom):
            root = (node.module or "").split(".")[0]
            if root in BANNED:
                raise SystemExit(f"blocked import: {node.module}")
        elif isinstance(node, ast.Call) and isinstance(node.func, ast.Name) and node.func.id in {"eval", "exec", "compile", "__import__"}:
            raise SystemExit(f"blocked call: {node.func.id}")


print("generated source\n")
print(SOLUTION_SRC)
check_source(SOLUTION_SRC)

with tempfile.TemporaryDirectory() as tmp:
    tmp_path = Path(tmp)
    (tmp_path / "solution.py").write_text(SOLUTION_SRC)
    (tmp_path / "test_parse_kv.py").write_text(TESTS)
    proc = subprocess.run(
        ["python", "-I", "test_parse_kv.py"],
        cwd=tmp,
        capture_output=True,
        text=True,
        timeout=8,
    )
print(proc.stdout)
if proc.returncode:
    print(proc.stderr)
    raise SystemExit(f"tests failed with exit {proc.returncode}")



## 4. Tool-use loop

The chat template renders `tools` under a `<tools>` block and expects calls as `<tool_call>name<arg_key>…</arg_key><arg_value>…</arg_value></tool_call>`. Thinking stays off so the first generate spends tokens on the call, not a trace. Stubs stand in for weather and local time; swap them for real APIs if you want.



In [ ]:
import json
from datetime import datetime
from zoneinfo import ZoneInfo

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather for a city.",
            "parameters": {
                "type": "object",
                "properties": {"city": {"type": "string", "description": "City name, e.g. Singapore"}},
                "required": ["city"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_local_time",
            "description": "Get the current local time in an IANA timezone.",
            "parameters": {
                "type": "object",
                "properties": {"timezone": {"type": "string", "description": "IANA timezone, e.g. Asia/Singapore"}},
                "required": ["timezone"],
            },
        },
    },
]


def get_weather(city: str) -> dict:
    return {"city": city, "celsius": 31, "condition": "partly cloudy"}


def get_local_time(timezone: str) -> dict:
    now = datetime.now(ZoneInfo(timezone))
    return {"timezone": timezone, "iso": now.isoformat(timespec="seconds")}


TOOL_FUNCTIONS = {"get_weather": get_weather, "get_local_time": get_local_time}


def parse_tool_calls(text):
    calls = []
    for block in re.finditer(r"<tool_call>(.*?)</tool_call>", text, flags=re.DOTALL):
        body = block.group(1)
        if "<arg_key>" in body:
            name, rest = body.split("<arg_key>", 1)
            args = {}
            for match in re.finditer(
                r"<arg_key>(.*?)</arg_key><arg_value>(.*?)</arg_value>",
                "<arg_key>" + rest,
                flags=re.DOTALL,
            ):
                key, value = match.group(1), match.group(2)
                try:
                    args[key] = json.loads(value)
                except json.JSONDecodeError:
                    args[key] = value
        else:
            name, args = body, {}
        calls.append({"name": name.strip(), "arguments": args})
    return calls


tool_messages = [{
    "role": "user",
    "content": (
        "I am in Singapore this afternoon. What is the weather, and what time is it there? "
        "Use the tools, then give a two-sentence recommendation."
    ),
}]
first = chat(tool_messages, tools=TOOLS, enable_thinking=False, max_new_tokens=256, temperature=0.4)
think, rest = split_think(first)
print("----- first generate -----")
print(first)

calls = parse_tool_calls(rest)
print("parsed calls:", calls)

if not calls:
    print("No <tool_call> block. The model answered in text, or the call was malformed.")
else:
    content = re.sub(r"<tool_call>.*?</tool_call>", "", rest, flags=re.DOTALL).strip()
    assistant = {
        "role": "assistant",
        "content": content,
        "tool_calls": [
            {"function": {"name": call["name"], "arguments": call["arguments"]}} for call in calls
        ],
    }
    if think:
        assistant["reasoning_content"] = think
    follow_messages = [*tool_messages, assistant]
    for call in calls:
        fn = TOOL_FUNCTIONS.get(call["name"])
        if fn is None:
            payload = {"error": f"unknown tool: {call['name']}"}
        else:
            try:
                payload = fn(**call["arguments"])
            except TypeError as exc:
                payload = {"error": str(exc)}
        print("tool", call["name"], payload)
        follow_messages.append({"role": "tool", "content": json.dumps(payload, separators=(",", ":"))})
    final = chat(follow_messages, tools=TOOLS, enable_thinking=False, max_new_tokens=220, temperature=0.4)
    _, final_answer = split_think(final)
    print("----- final -----")
    print(final_answer or final)



## 5. Caveats

- **Custom modeling code.** The checkpoint is `Spark2_5ForCausalLM`. Loading it runs `modeling_spark.py` and `configuration_spark.py` from the Hub (`trust_remote_code=True`). Read those files before you use this in anything that is not a notebook.
- **Pinned transformers.** The card requires `transformers==4.57.1`. Colab's preinstalled copy is newer; the install cell downgrades it, and a runtime restart is what makes the pin take effect. Do not bump the version in this notebook unless the card changes.
- **fp16 overflow on a T4.** The saved dtype is bfloat16. A T4 has no native bfloat16, so the load cell uses float16 and falls back to 4-bit bitsandbytes if load or generate overflows. Newer GPUs (L4, A100) take bfloat16, the card default.
- **Modest generation length.** The card's snippet uses `max_new_tokens=131072` and advertises a 1,048,576-token context. Neither fits a 16 GB T4. Demos here stay in the low hundreds of new tokens, with `reasoning_effort="low"` when thinking is on.
- **Optional GGUF.** [`XHToken/Spark-X2.6-4B-GGUF`](https://huggingface.co/XHToken/Spark-X2.6-4B-GGUF) has `Spark-X2.6-4B-Q4_K_M.gguf` (~3.54 GB). Native `spark2_5` support needs [llama.cpp](https://github.com/ggml-org/llama.cpp/) `b10828` or later. Prefer the transformers float16 / 4-bit path above if it fits; unload that model before you download the GGUF on a 16 GB card.

